# FSG 월간보고 데이터 만들기 — Drive 폴더 하나로 (PC 없이, Colab)

Drive 폴더 **`CRM_원본`** 에 CRM 내보내기 파일을 **이름 그대로** 넣어 두면, 이 노트북이 최신 파일을 골라 새 엔진(fsg-report)으로 보고서 숫자를 계산하고
결과를 Drive 폴더 `FSG_report_out/<기간>/` 에 저장합니다. Google 시트는 쓰지 않습니다. 데이터는 내 Google 계정 안(Colab)에서만 처리됩니다.

| 넣을 파일 | 파일 이름에 들어 있어야 하는 말 | 없으면 |
|---|---|---|
| CRM 「2. 모든 약속/활동」 xlsx | `약속_활동` | 실행 안 됨(필수) |
| CRM 「Serial No」 자산대장 xlsx | `Serial` | 운영건전성·집중관리대상·설치 현황 빠짐 |
| 설치 공유캘린더 내보내기 (`시스템설치_공유캘린더_YYYYMMDD.xlsx`) | `캘린더` | 4장 설치 현황 빠짐 |
| 기관명매핑 (캘린더 표기 \| CRM 표기 \| 비고) | `기관명매핑` | 자동 매핑 제안만 |
| 설치_수동입력 (시작일 \| 기관명 \| 캘린더 표기 \| 모델1 수량1 …) | `수동입력` | 수동 설치 없음 |

- 지난달 파일을 지울 필요 없습니다. 같은 종류가 여러 개면 **파일명 날짜 → 수정 시각**으로 최신 것을 씁니다.
- 기관명매핑·설치_수동입력은 폴더 안에 Google 시트로 두어도 됩니다(첫 번째 탭을 읽음).
- 로스터는 저장소의 `fsg-kpi.html`(대시보드와 같은 명단)을 씁니다.

**순서**: 메뉴 **런타임 › 모두 실행** → 권한 창에서 허용 → 2~4분 뒤 맨 아래 결과 확인.
결과: `report-data_<기간>.json`(PPT 빌더 입력) · `검산_<기간>.xlsx`(시트 정제 탭과 같은 배치) · `요약_<기간>.txt`(이름 없는 요약 — Claude가 Drive 커넥터로 읽음).

In [ ]:
#@title 1) 설정 — 보통 그대로 둡니다
IN_FOLDER  = 'CRM_원본'        #@param {type:"string"}  (CRM 파일을 넣는 Drive 폴더 이름)
OUT_FOLDER = 'FSG_report_out'  #@param {type:"string"}  (결과를 저장할 Drive 폴더 이름)
BRANCH     = 'claude/fsg-report-engine'  #@param {type:"string"}
PERIOD     = ''   #@param {type:"string"}  (비우면 지난달. 예: 2026-09, 2026-07~2026-09, FY2026Q2)
CUT        = ''   #@param {type:"string"}  (CRM 추출일. 비우면 활동 파일명의 날짜. 예: 2026-10-12)
REPO_URL = 'https://github.com/balhm12/Sysmex-schedule.git'
IN_DIR, OUT_DIR, REPO = '/content/crm_in', '/content/out', '/content/sysmex-schedule'
import datetime
KST = datetime.timezone(datetime.timedelta(hours=9))
if not PERIOD:
    first = datetime.datetime.now(KST).replace(day=1)
    PERIOD = (first - datetime.timedelta(days=1)).strftime('%Y-%m')
print('기간:', PERIOD, '· 추출일:', CUT or '(활동 파일명 날짜)')

In [ ]:
#@title 2) Drive 폴더에서 CRM 파일 내려받기 (권한 창이 뜨면 허용)
import io, os, re, shutil
from google.colab import auth
auth.authenticate_user()
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
creds, _ = default()
drv = build('drive', 'v3', credentials=creds)
FOLDER_MIME, GSHEET_MIME = 'application/vnd.google-apps.folder', 'application/vnd.google-apps.spreadsheet'
XLSX_MIME = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
def find_folder(name, parent=None):
    q = f"name='{name}' and mimeType='{FOLDER_MIME}' and trashed=false" + (f" and '{parent}' in parents" if parent else '')
    f = drv.files().list(q=q, fields='files(id,name)', pageSize=5, supportsAllDrives=True, includeItemsFromAllDrives=True).execute().get('files', [])
    return f[0]['id'] if f else None
fid = find_folder(IN_FOLDER)
assert fid, f"Drive에 '{IN_FOLDER}' 폴더가 없습니다. 내 드라이브에 만들고 CRM 내보내기 파일을 넣어 주세요."
files, tok = [], None
while True:
    r = drv.files().list(q=f"'{fid}' in parents and trashed=false", fields='nextPageToken,files(id,name,mimeType,size,modifiedTime)',
                         pageSize=200, pageToken=tok, supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
    files += r.get('files', []); tok = r.get('nextPageToken')
    if not tok: break
shutil.rmtree(IN_DIR, ignore_errors=True); os.makedirs(IN_DIR)
got, skipped = 0, 0
for f in files:
    name, mt = f['name'], f['mimeType']
    if mt == GSHEET_MIME:
        req, name = drv.files().export_media(fileId=f['id'], mimeType=XLSX_MIME), re.sub(r'\.xlsx$', '', name) + '.xlsx'
    elif re.search(r'\.(xlsx|xlsm|xls|csv)$', name, re.I) and mt != FOLDER_MIME:
        req = drv.files().get_media(fileId=f['id'], supportsAllDrives=True)
    else:
        skipped += 1; continue
    buf = io.BytesIO(); dl = MediaIoBaseDownload(buf, req)
    done = False
    while not done: _, done = dl.next_chunk()
    with open(os.path.join(IN_DIR, name), 'wb') as w: w.write(buf.getvalue())
    got += 1
    # 수정 시각을 Drive와 맞춘다(같은 날짜 파일이 여럿일 때 최신 선택용)
    ts = datetime.datetime.fromisoformat(f['modifiedTime'].replace('Z', '+00:00')).timestamp()
    os.utime(os.path.join(IN_DIR, name), (ts, ts))
    print(f"  내려받음: {name} ({int(f.get('size', 0) or 0)//1024} KB)")
print(f'파일 {got}개 내려받음 · 건너뜀 {skipped}개 (xlsx/csv/Google 시트만 사용)')
assert got, '폴더가 비어 있습니다.'

In [ ]:
#@title 3) 엔진 받기 → 보고서 데이터 계산
import subprocess
def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return r.returncode, (r.stdout or '') + (r.stderr or '')
print(run('node -v')[1].strip(), '/ npm', run('npm -v')[1].strip())
shutil.rmtree(REPO, ignore_errors=True)
rc, out = run(f'git clone -q -b {BRANCH} {REPO_URL} {REPO}'); assert rc == 0, out
rc, out = run('npm install --no-audit --no-fund --silent', cwd=REPO); assert rc == 0, out
shutil.rmtree(OUT_DIR, ignore_errors=True); os.makedirs(OUT_DIR)
cut = f' --cut "{CUT}"' if CUT else ''
rc, out = run(f'node fsg-report/report-data.js --period "{PERIOD}" --dir "{IN_DIR}" --out "{OUT_DIR}"{cut}', cwd=REPO)
print(out)
assert rc == 0, '계산 실패 — 위 메시지를 확인해 주세요(보통 활동 파일이 없거나 열 이름이 다를 때)'
import json, glob
JSON_PATH = glob.glob(f'{OUT_DIR}/report-data_*.json')[0]
STEM = os.path.basename(JSON_PATH)[len('report-data_'):-len('.json')]
mf = json.load(open(JSON_PATH, encoding='utf-8'))['meta']['files']
used = ' · '.join(f"{k} {v or '(없음)'}" for k, v in [('활동', mf['activity']), ('Serial', mf['serial']), ('캘린더', mf['calendar']), ('기관명매핑', mf['acctMap']), ('수동입력', mf['manual'])])
summary = f'[사용한 파일] {used}\n\n' + out
SUMMARY_PATH = f'{OUT_DIR}/요약_{STEM}.txt'
open(SUMMARY_PATH, 'w', encoding='utf-8').write(summary)
print('[사용한 파일]', used)

In [ ]:
#@title 4) 결과를 Drive 폴더 FSG_report_out/<기간>/ 에 저장 (같은 이름이 있으면 새 버전으로 갱신)
from googleapiclient.http import MediaFileUpload
def ensure_folder(name, parent=None):
    f = find_folder(name, parent)
    if f: return f
    body = {'name': name, 'mimeType': FOLDER_MIME}
    if parent: body['parents'] = [parent]
    return drv.files().create(body=body, fields='id').execute()['id']
root = ensure_folder(OUT_FOLDER); sub = ensure_folder(STEM, root)
def upload(path, mime):
    name = os.path.basename(path)
    q = f"name='{name}' and '{sub}' in parents and trashed=false"
    ex = drv.files().list(q=q, fields='files(id)', pageSize=1).execute().get('files', [])
    media = MediaFileUpload(path, mimetype=mime, resumable=True)
    if ex: f = drv.files().update(fileId=ex[0]['id'], media_body=media, fields='id,name,webViewLink').execute(); how = '갱신'
    else: f = drv.files().create(body={'name': name, 'parents': [sub]}, media_body=media, fields='id,name,webViewLink').execute(); how = '저장'
    print(f"  {how}: {OUT_FOLDER}/{STEM}/{f['name']}  {f['webViewLink']}")
upload(SUMMARY_PATH, 'text/plain')
upload(JSON_PATH, 'application/json')
upload(glob.glob(f'{OUT_DIR}/검산_*.xlsx')[0], XLSX_MIME)
print('→ Claude에게 "보고 데이터 만들었어"라고 하면 Drive의 요약(이름 없음)을 읽어 확인합니다. 보고서(PPT·Word)는 검산_<기간>.xlsx 를 올리면 됩니다.')